In [1]:
from agents import *
from routing import distrito_tec, get_closest_place_node_id
import numpy as np
import random

sub_graph, routable_restaurants, residential_zones = distrito_tec()


In [2]:
routable_restaurants.head(5)

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
1,node,890657529,POINT (-100.29377 25.65441),restaurant,NaN,NaN,NaN,NaN,Costeñito,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
2,node,890657855,POINT (-100.29355 25.65472),restaurant,NaN,NaN,NaN,NaN,Wings Army,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
3,node,890666591,POINT (-100.28436 25.64771),restaurant,NaN,NaN,NaN,NaN,Manhattan,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09
4,node,890667172,POINT (-100.28432 25.64783),bar,NaN,NaN,NaN,NaN,La Rambla,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09


In [3]:
routable_restaurants.shape

(50, 40)

In [4]:
residential_zones.head(5)

,element,id,geometry,landuse,name,place,addr:city,addr:street,building:levels,residential,operator,type
0,relation,9463437,"POLYGON ((-100.29348 25.66187, -100.2936 25.66...",residential,La Florida,NaN,NaN,NaN,NaN,NaN,NaN,multipolygon
1,way,163034600,"POLYGON ((-100.27483 25.62377, -100.27426 25.6...",residential,Contry,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,way,163034602,"POLYGON ((-100.27739 25.64568, -100.27511 25.6...",residential,Contry Tesoro,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,way,163034603,"POLYGON ((-100.28282 25.64297, -100.28238 25.6...",residential,Contry Lux,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN
4,way,163034605,"POLYGON ((-100.28052 25.64492, -100.27925 25.6...",residential,Las Musas,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
routable_restaurants.iloc[[0]]

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09


In [6]:
# =========================================================
# SCENARIO SETUP (users, drivers, restaurants, demand rate)
# =========================================================

# ---- PARAMETERS — edit these to run different scenarios ----
N_USERS          = 10000
N_DRIVERS        = 200
ORDER_RATE       = 40          # orders per minute (Poisson lambda)
DISPATCH_MODE    = 'hungarian' # 'greedy' or 'hungarian'
DISPATCH_INTERVAL = 15         # seconds between batch dispatches (hungarian only)

# ---------------------------------------------------------
# 1. Initialize simulation
# ---------------------------------------------------------

sim = Simulation(
    graph=sub_graph,
    step_size=10,
    dispatch_mode=DISPATCH_MODE,
    dispatch_interval=DISPATCH_INTERVAL,
)

# ---------------------------------------------------------
# 2. Add restaurants
#    Ratings are sampled uniformly [3.0, 5.0] so the MNL
#    restaurant-choice model has something to work with.
# ---------------------------------------------------------

for i in range(len(routable_restaurants)):

    res_node = get_closest_place_node_id(
        routable_restaurants.iloc[[i]],
        sub_graph
    )

    res = Restaurant(
        restaurant_id=i,
        location=res_node,
        rating=round(random.uniform(3.0, 5.0), 1),  # heterogeneous ratings
        capacity=10,
        avg_prep_time=300,
        service_radius=50000,
    )

    sim.add_restaurant(res)

print(f"{len(sim.restaurants)} restaurants loaded")

# ---------------------------------------------------------
# 3. Generate residential nodes for users / drivers
# ---------------------------------------------------------

sampled_zones = residential_zones.sample(N_USERS, replace=True)

residential_nodes = [
    get_closest_place_node_id(sampled_zones.iloc[[i]], sub_graph)
    for i in range(N_USERS)
]

# ---------------------------------------------------------
# 4. Create users
# ---------------------------------------------------------

for i in range(N_USERS):
    sim.add_user(User(user_id=i, location=residential_nodes[i]))

print(f"{len(sim.users)} users created")

# ---------------------------------------------------------
# 5. Create drivers
# ---------------------------------------------------------

for i in range(N_DRIVERS):
    sim.add_driver(Driver(
        driver_id=i,
        location_node=random.choice(residential_nodes),
        speed=None,   # sampled from truncated normal in Driver.__init__
    ))

print(f"{len(sim.drivers)} drivers created")
print(f"Dispatch mode : {DISPATCH_MODE}")
print("Simulation ready — orders generated dynamically each tick.")


50 restaurants loaded
10000 users created
200 drivers created
Dispatch mode : hungarian
Simulation ready — orders generated dynamically each tick.


In [7]:
SHOW_TRAILS = False

In [8]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import numpy as np
import osmnx as ox
from IPython.display import HTML

# --- Map canvas ---
fig, ax = ox.plot_graph(
    sub_graph, bgcolor='k', edge_color='#333333',
    node_size=0, edge_linewidth=0.5, show=False, close=False
)

# --- Static restaurant markers ---
restaurant_lons = [sub_graph.nodes[r.location]['x'] for r in sim.restaurants.values()]
restaurant_lats = [sub_graph.nodes[r.location]['y'] for r in sim.restaurants.values()]

ax.scatter(
    restaurant_lons, restaurant_lats,
    s=120, c='red', marker='^',
    edgecolors='white', linewidth=1.2,
    zorder=12, label='Restaurants'
)

# --- Driver visual elements ---
drivers_to_watch = list(sim.drivers.values())
if not drivers_to_watch:
    raise ValueError("No drivers found — check setup cell.")

colors = cm.rainbow(np.linspace(0, 1, len(drivers_to_watch)))
dots = ax.scatter([], [], c=[], s=60, zorder=10, edgecolors='white', linewidth=0.8)

if SHOW_TRAILS:
    lines = [ax.plot([], [], color=colors[i], lw=2.5, alpha=0.7, zorder=9)[0]
             for i in range(len(drivers_to_watch))]
else:
    lines = []

trail_history = [[] for _ in drivers_to_watch] if SHOW_TRAILS else None

# HUD text (top-left)
time_text = ax.text(
    0.02, 0.95, '', transform=ax.transAxes,
    color='white', fontsize=11, fontweight='bold',
    verticalalignment='top', fontfamily='monospace'
)

# --- Animation update ---
def update(frame):
    # Single generate_orders call per frame (was incorrectly called twice before)
    generate_orders(sim, rate_per_minute=ORDER_RATE)
    sim.run_tick()

    # Driver positions
    current_lons, current_lats = [], []
    for i, driver in enumerate(drivers_to_watch):
        if driver.coords != (0.0, 0.0):
            lon, lat = driver.coords[1], driver.coords[0]
        else:
            node_data = sub_graph.nodes[driver.location]
            lon, lat = node_data['x'], node_data['y']

        current_lons.append(lon)
        current_lats.append(lat)

        if SHOW_TRAILS:
            trail_history[i].append((lon, lat))
            h_x, h_y = zip(*trail_history[i])
            lines[i].set_data(h_x, h_y)

    dots.set_offsets(np.c_[current_lons, current_lats])
    dots.set_color(colors)

    # HUD — pull from metrics_snapshot so counts are always consistent
    m = sim.metrics_snapshot()
    s = m['orders_by_status']
    hud = (
        f"t={int(sim.current_time)}s  [{m['dispatch_mode']}]\n"
        f"PREP:{s['PREPARING']}  READY:{s['READY']}  "
        f"PICKUP:{s['PICKED_UP']}  DONE:{s['DELIVERED']}\n"
        f"idle drivers:{m['idle_drivers']}  "
        f"avg e2e:{int(m['avg_end_to_end_s'] or 0)}s"
    )
    time_text.set_text(hud)

    return ([dots, time_text] + lines) if SHOW_TRAILS else [dots, time_text]

# frames=300 * step_size=10 => 3000 simulated seconds (~50 min)
ani = FuncAnimation(fig, update, frames=300, interval=500, blit=True, repeat=False)
plt.close()
ani.save("sim.mp4")


In [11]:
# =========================================================
# POST-RUN METRICS
# Run this cell after the animation finishes (or after
# sim.run_until(T)) to inspect KPIs for the current run.
# =========================================================

import pandas as pd

m = sim.metrics_snapshot()
print(f"Dispatch mode      : {m['dispatch_mode']}")
print(f"Simulated time     : {m['time']:.0f}s")
print(f"Total orders       : {m['total_orders']}")
print(f"Delivered          : {m['n_delivered']}")
print(f"Pending unassigned : {m['pending_unassigned']}")
print(f"Idle drivers       : {m['idle_drivers']}")
print()
print(f"Avg end-to-end     : {m['avg_end_to_end_s']:.1f}s" if m['avg_end_to_end_s'] else "Avg end-to-end : n/a")
print(f"Avg food wait      : {m['avg_food_wait_s']:.1f}s"  if m['avg_food_wait_s']   else "Avg food wait  : n/a")
print(f"Avg dispatch delay : {m['avg_dispatch_delay_s']:.1f}s" if m['avg_dispatch_delay_s'] else "Avg dispatch delay : n/a")

# Per-order dataframe for deeper analysis
records = [
    {
        'order_id'        : o.id,
        'restaurant_id'   : o.restaurant_id,
        'start_time'      : o.start_time,
        'assigned_time'   : o.assigned_time,
        'pickup_time'     : o.pickup_time,
        'delivered_time'  : o.delivered_time,
        'end_to_end_s'    : o.end_to_end_time,
        'food_wait_s'     : o.food_wait_time,
        'dispatch_delay_s': o.dispatch_delay,
        'status'          : o.status,
        'user_id'         : o.user_id,
        'driver_id'       : o.driver_id,
    }
    for o in sim.orders.values()
]

df = pd.DataFrame(records)
df.head(10)


Dispatch mode      : hungarian
Simulated time     : 3030s
Total orders       : 1311
Delivered          : 944
Pending unassigned : 181
Idle drivers       : 200

Avg end-to-end     : 616.2s
Avg food wait      : 53.9s
Avg dispatch delay : 111.9s


,order_id,restaurant_id,start_time,assigned_time,pickup_time,delivered_time,end_to_end_s,food_wait_s,dispatch_delay_s,status,user_id,driver_id
0,1,41,0.0,20.0,250.0,640.0,640.0,109.339669,20.0,DELIVERED,3922,86.0
1,2,25,0.0,20.0,170.0,690.0,690.0,0.000000,20.0,DELIVERED,2724,48.0
2,3,31,0.0,20.0,230.0,790.0,790.0,0.000000,20.0,DELIVERED,5164,54.0
3,4,31,0.0,20.0,270.0,820.0,820.0,73.342174,20.0,DELIVERED,8856,25.0
4,5,9,0.0,20.0,200.0,470.0,470.0,65.524333,20.0,DELIVERED,8049,177.0
5,6,5,0.0,20.0,120.0,190.0,190.0,0.000000,20.0,DELIVERED,1358,28.0
6,7,24,10.0,20.0,190.0,480.0,470.0,0.000000,10.0,DELIVERED,731,92.0
7,8,40,10.0,20.0,90.0,270.0,260.0,0.000000,10.0,DELIVERED,9774,114.0
8,9,5,10.0,20.0,190.0,360.0,350.0,0.000000,10.0,DELIVERED,226,32.0
9,10,29,10.0,20.0,190.0,640.0,630.0,0.000000,10.0,DELIVERED,6012,87.0


In [12]:
df.to_csv("results.csv",index=0)

In [13]:
df

,order_id,restaurant_id,start_time,assigned_time,pickup_time,delivered_time,end_to_end_s,food_wait_s,dispatch_delay_s,status,user_id,driver_id
0,1,41,0.0,20.0,250.0,640.0,640.0,109.339669,20.0,DELIVERED,3922,86.0
1,2,25,0.0,20.0,170.0,690.0,690.0,0.000000,20.0,DELIVERED,2724,48.0
2,3,31,0.0,20.0,230.0,790.0,790.0,0.000000,20.0,DELIVERED,5164,54.0
3,4,31,0.0,20.0,270.0,820.0,820.0,73.342174,20.0,DELIVERED,8856,25.0
4,5,9,0.0,20.0,200.0,470.0,470.0,65.524333,20.0,DELIVERED,8049,177.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1306,1307,11,3010.0,3020.0,NaN,NaN,NaN,NaN,10.0,PREPARING,3419,61.0
1307,1308,21,3010.0,NaN,NaN,NaN,NaN,NaN,NaN,PREPARING,7167,NaN
1308,1309,0,3020.0,NaN,NaN,NaN,NaN,NaN,NaN,PREPARING,3292,NaN
1309,1310,11,3020.0,NaN,NaN,NaN,NaN,NaN,NaN,PREPARING,7586,NaN
